In [1]:
from pyspark.sql import SparkSession
import os
from dotenv import load_dotenv

load_dotenv()

AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
S3_BUCKET_NAME = os.getenv("S3_BUCKET_NAME")

# Tạo đường dẫn S3: s3a://bucket-name/bronze/
S3_BRONZE_PATH = f"s3a://{S3_BUCKET_NAME}/bronze/"
LOCAL_DATA_DIR = "/opt/spark/dataset/"

spark = SparkSession.builder \
    .appName("Ingestion_Bronze_Layer") \
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print(f"Spark Session đã sẵn sàng!")
print(f"S3 Bronze Path: {S3_BRONZE_PATH}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/15 10:44:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/15 10:44:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark Session đã sẵn sàng!
S3 Bronze Path: s3a://olist-lakehouse-2026/bronze/


### Nạp toàn bộ dữ liệu lần đầu (Full Load) cho tập CSV

In [4]:
# Danh sách tên các bảng Olist
olist_tables = [
    "olist_customers_dataset",
    "olist_geolocation_dataset",
    "olist_orders_dataset",
    "olist_order_items_dataset",
    "olist_order_payments_dataset",
    "olist_order_reviews_dataset",
    "olist_products_dataset",
    "olist_sellers_dataset",
    "product_category_name_translation"
]

# Lưu trữ số dòng mỗi bảng để kiểm định sau
full_load_counts = {}

print("Bắt đầu quá trình nạp Full Load 9 bảng Olist lên Bronze Layer\n")

for table_name in olist_tables:
    csv_file_path = f"{LOCAL_DATA_DIR}{table_name}.csv"
    s3_target_path = f"{S3_BRONZE_PATH}{table_name}"
    
    try:
        print(f"Đang xử lý bảng: {table_name}...")
        
        df = spark.read.csv(
            csv_file_path, 
            header=True, 
            inferSchema=True, 
            multiLine=True, 
            escape='"'
        )
        row_count = df.count()
        full_load_counts[table_name] = row_count
        
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .save(s3_target_path)
            
        print(f"Đã đẩy {row_count} dòng lên: {s3_target_path}")
        
    except Exception as e:
        print(f"Lỗi khi xử lý bảng {table_name}: {e}")

print("\nHoàn tất quá trình nạp Full Load dữ liệu Olist!")

Bắt đầu quá trình nạp Full Load 9 bảng Olist lên Bronze Layer

Đang xử lý bảng: olist_customers_dataset...


Đã đẩy 99441 dòng lên: s3a://olist-lakehouse-2026/bronze/olist_customers_dataset
Đang xử lý bảng: olist_geolocation_dataset...


Đã đẩy 1000163 dòng lên: s3a://olist-lakehouse-2026/bronze/olist_geolocation_dataset
Đang xử lý bảng: olist_orders_dataset...


Đã đẩy 99441 dòng lên: s3a://olist-lakehouse-2026/bronze/olist_orders_dataset
Đang xử lý bảng: olist_order_items_dataset...


Đã đẩy 112650 dòng lên: s3a://olist-lakehouse-2026/bronze/olist_order_items_dataset
Đang xử lý bảng: olist_order_payments_dataset...


Đã đẩy 103886 dòng lên: s3a://olist-lakehouse-2026/bronze/olist_order_payments_dataset
Đang xử lý bảng: olist_order_reviews_dataset...


Đã đẩy 99224 dòng lên: s3a://olist-lakehouse-2026/bronze/olist_order_reviews_dataset
Đang xử lý bảng: olist_products_dataset...


Đã đẩy 32951 dòng lên: s3a://olist-lakehouse-2026/bronze/olist_products_dataset
Đang xử lý bảng: olist_sellers_dataset...


Đã đẩy 3095 dòng lên: s3a://olist-lakehouse-2026/bronze/olist_sellers_dataset
Đang xử lý bảng: product_category_name_translation...


Đã đẩy 71 dòng lên: s3a://olist-lakehouse-2026/bronze/product_category_name_translation

Hoàn tất quá trình nạp Full Load dữ liệu Olist!


### Nạp Gia Tăng (Incremental Load)

In [2]:
from delta.tables import DeltaTable

print("=" * 60)
print("BẮT ĐẦU CHẠY LUỒNG NẠP GIA TĂNG (INCREMENTAL LOAD)")
print("=" * 60)

# Thư mục chứa file dữ liệu mới
INCREMENTAL_DATA_DIR = "/opt/spark/dataset/incremental/"

# Danh sách các bảng hỗ trợ nạp gia tăng cùng khóa chính
incremental_tables = {
    "olist_customers_dataset": "customer_id",
    "olist_orders_dataset": "order_id",
    "olist_order_items_dataset": ["order_id", "order_item_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
    "olist_order_reviews_dataset": "review_id",
    "olist_products_dataset": "product_id",
    "olist_sellers_dataset": "seller_id",
    "olist_geolocation_dataset": ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"],
    "product_category_name_translation": "product_category_name",
    "holidays_dataset": "date"  # Khóa chính là ngày lễ
}

incremental_results = []

for table_name, merge_key in incremental_tables.items():
    new_data_file = f"{INCREMENTAL_DATA_DIR}{table_name}_new.csv"
    bronze_path = f"{S3_BRONZE_PATH}{table_name}"
    
    try:
        print(f"\n[{table_name}]")
        print(f"Đang kiểm tra file mới: {new_data_file}")
        
        df_new = spark.read.csv(new_data_file, header=True, inferSchema=True)
        new_count = df_new.count()
        
        if new_count == 0:
            print(f"Không có dữ liệu mới. Bỏ qua.")
            continue
            
        print(f"Đã tìm thấy {new_count} bản ghi mới.")
        
        if DeltaTable.isDeltaTable(spark, bronze_path):
            delta_table = DeltaTable.forPath(spark, bronze_path)
            
            if isinstance(merge_key, list):
                merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in merge_key])
            else:
                merge_condition = f"target.{merge_key} = source.{merge_key}"
            
            delta_table.alias("target").merge(
                df_new.alias("source"),
                merge_condition
            ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
            
            print(f"MERGE thành công {new_count} bản ghi.")
        else:
            df_new.write.format("delta").mode("overwrite").save(bronze_path)
            print(f"Tạo mới Delta Table với {new_count} bản ghi.")
        
        total_count = spark.read.format("delta").load(bronze_path).count()
        incremental_results.append((table_name, new_count, total_count))
        print(f"Tổng số bản ghi trên Bronze Layer: {total_count}")
        
    except Exception as e:
        if "Path does not exist" in str(e) or "Unable to infer schema" in str(e):
            print(f"Không tìm thấy file dữ liệu mới. Bỏ qua.")
        else:
            print(f"Lỗi: {e}")

print("\n" + "=" * 60)
print("KẾT QUẢ NẠP GIA TĂNG:")
print("=" * 60)
if incremental_results:
    for table, new_cnt, total_cnt in incremental_results:
        print(f"{table}: +{new_cnt} → Tổng: {total_cnt}")
else:
    print(" Không có dữ liệu mới để nạp.")

BẮT ĐẦU CHẠY LUỒNG NẠP GIA TĂNG (INCREMENTAL LOAD)

[olist_customers_dataset]
Đang kiểm tra file mới: /opt/spark/dataset/incremental/olist_customers_dataset_new.csv
Đã tìm thấy 2 bản ghi mới.


26/04/03 11:21:59 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/04/03 11:22:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

MERGE thành công 2 bản ghi.


Tổng số bản ghi trên Bronze Layer: 99443

[olist_orders_dataset]
Đang kiểm tra file mới: /opt/spark/dataset/incremental/olist_orders_dataset_new.csv
Không tìm thấy file dữ liệu mới. Bỏ qua.

[olist_order_items_dataset]
Đang kiểm tra file mới: /opt/spark/dataset/incremental/olist_order_items_dataset_new.csv
Không tìm thấy file dữ liệu mới. Bỏ qua.

[olist_order_payments_dataset]
Đang kiểm tra file mới: /opt/spark/dataset/incremental/olist_order_payments_dataset_new.csv
Đã tìm thấy 3 bản ghi mới.


MERGE thành công 3 bản ghi.


Tổng số bản ghi trên Bronze Layer: 103889

[olist_order_reviews_dataset]
Đang kiểm tra file mới: /opt/spark/dataset/incremental/olist_order_reviews_dataset_new.csv
Không tìm thấy file dữ liệu mới. Bỏ qua.

[olist_products_dataset]
Đang kiểm tra file mới: /opt/spark/dataset/incremental/olist_products_dataset_new.csv
Không tìm thấy file dữ liệu mới. Bỏ qua.

[olist_sellers_dataset]
Đang kiểm tra file mới: /opt/spark/dataset/incremental/olist_sellers_dataset_new.csv
Không tìm thấy file dữ liệu mới. Bỏ qua.

[olist_geolocation_dataset]
Đang kiểm tra file mới: /opt/spark/dataset/incremental/olist_geolocation_dataset_new.csv
Không tìm thấy file dữ liệu mới. Bỏ qua.

[product_category_name_translation]
Đang kiểm tra file mới: /opt/spark/dataset/incremental/product_category_name_translation_new.csv
Không tìm thấy file dữ liệu mới. Bỏ qua.

[holidays_dataset]
Đang kiểm tra file mới: /opt/spark/dataset/incremental/holidays_dataset_new.csv
Không tìm thấy file dữ liệu mới. Bỏ qua.

KẾT QUẢ NẠP GIA

### Kiểm định dữ liệu (Data Validation)

In [3]:
print("=" * 70)
print("KIỂM ĐỊNH DỮ LIỆU - ĐỐI CHIẾU SOURCE VS BRONZE LAYER")
print("=" * 70)

validation_passed = True
validation_results = []

# Danh sách tất cả các bảng cần kiểm định (bao gồm cả holidays từ API)
all_tables = olist_tables + ["holidays_dataset"]

for table_name in all_tables:
    s3_path = f"{S3_BRONZE_PATH}{table_name}"
    
    try:
        # Lấy số dòng đã lưu từ full_load_counts (lúc nạp)
        source_count = full_load_counts.get(table_name, -1)
        
        # Đọc lại từ S3 Bronze và đếm
        df_bronze = spark.read.format("delta").load(s3_path)
        bronze_count = df_bronze.count()
        
        # So sánh
        if source_count == bronze_count:
            status = "✓ PASS"
        elif source_count == -1:
            status = "⚠ SKIP (không có source count)"
        else:
            status = "✗ FAIL"
            validation_passed = False
        
        validation_results.append({
            "table": table_name,
            "source": source_count,
            "bronze": bronze_count,
            "status": status
        })
        
    except Exception as e:
        validation_results.append({
            "table": table_name,
            "source": "N/A",
            "bronze": "ERROR",
            "status": f"✗ ERROR: {str(e)[:50]}"
        })
        validation_passed = False

# In kết quả dạng bảng
print(f"{'Bảng':<45} {'Source':<12} {'Bronze':<12} {'Kết quả'}")
print("-" * 90)

for r in validation_results:
    print(f"{r['table']:<45} {str(r['source']):<12} {str(r['bronze']):<12} {r['status']}")

print("-" * 90)
if validation_passed:
    print("\nKIỂM ĐỊNH THÀNH CÔNG: Tất cả dữ liệu đã được nạp chính xác!")
else:
    print("\nCẢNH BÁO: Có lỗi trong quá trình kiểm định, vui lòng kiểm tra!")

KIỂM ĐỊNH DỮ LIỆU - ĐỐI CHIẾU SOURCE VS BRONZE LAYER


26/04/11 09:14:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Bảng                                          Source       Bronze       Kết quả
------------------------------------------------------------------------------------------
olist_customers_dataset                       99441        99441        ✓ PASS
olist_geolocation_dataset                     1000163      1000163      ✓ PASS
olist_orders_dataset                          99441        99441        ✓ PASS
olist_order_items_dataset                     112650       112650       ✓ PASS
olist_order_payments_dataset                  103886       103886       ✓ PASS
olist_order_reviews_dataset                   104162       104162       ✓ PASS
olist_products_dataset                        32951        32951        ✓ PASS
olist_sellers_dataset                         3095         3095         ✓ PASS
product_category_name_translation             71           71           ✓ PASS
holidays_dataset                              N/A          ERROR        ✗ ERROR: [PATH_NOT_FOUND] Path does not exist: